# Coarse refinement: three-set analysis (iteration 28)

Reads `refine_<set>.csv` from `Coarse_Refine.ipynb` (per query: coarse peak `c`, parabola `p`, finer level 2 / 4 levels `f2`, `f4` and with parabola `f2p`, `f4p`; error in sigma; coarse z). Photo level = the query with the highest coarse z.
Part 1: summary per set. Part 2: does averaging the refined peaks of agreeing queries help now that the grid error is smaller (it did not for the coarse peaks, iteration 27)? Part 3: the kill rule.

In [ ]:
import numpy as np, pandas as pd
R = "../results/coarse_refine/"
def pts(e): return np.where(e<=1,10,np.where(e<=2,6,np.where(e<=4,3,np.where(e<=8,0,-20))))
names = {"new (332)": "highalt_dev332", "official (500)": "official500", "NUDT (300)": "nudt300"}
tabs = {n: pd.read_csv(R + "refine_%s.csv" % t) for n, t in names.items()}
variants = ["c", "p", "f2", "f2p", "f4", "f4p"]
def best_rows(t): return t.loc[t.groupby("photo").z1.idxmax()]
def summary(frame, label):
    out = []
    for v in variants:
        if "err_" + v not in frame: continue
        e = frame["err_" + v].dropna()
        # photos without that variant (no finer level in the store) keep their coarse answer
        e_full = frame["err_" + v].fillna(frame["err_c"])
        out.append(dict(variant=v, n_have=len(e), within1=round((e_full<=1).mean(),3), within2=round((e_full<=2).mean(),3), wrong=int((e_full>8).sum()),
                        median_not_wrong=round(e_full[e_full<=8].median(),2), pts_per_photo=round(pts(e_full.values).mean(),2)))
    print("\n==", label, "(n =", len(frame), ")"); print(pd.DataFrame(out).to_string(index=False))
for n, t in tabs.items():
    b = best_rows(t); summary(b, n + ", all heights")
    if (b.height >= 800).any(): summary(b[b.height >= 800], n + ", above 800 m")

## Averaging the refined peaks of queries within r sigma of the best-z answer

In [ ]:
def averaged(t, variant, r):
    rows = []
    for p, g in t.groupby("photo"):
        g = g.dropna(subset=["x_" + variant]); 
        if len(g) == 0: continue
        b = g.loc[g.z1.idxmax()]; d = np.hypot(g["x_"+variant]-b["x_"+variant], g["y_"+variant]-b["y_"+variant])/b.sigma_m; near = g[d <= r]
        x, y = near["x_"+variant].mean(), near["y_"+variant].mean()
        rows.append(dict(photo=p, height=b.height, err=np.hypot(x-b.tx, y-b.ty)/b.sigma_m, single=b["err_"+variant], votes=len(near)))
    return pd.DataFrame(rows)
for n, t in tabs.items():
    for v in ["p", "f2p", "f4p"]:
        if "x_" + v not in t: continue
        for r in (1, 2):
            a = averaged(t, v, r); a = a[a.height >= 800] if n.startswith("new") else a
            print(n, v, "r=%d" % r, "n", len(a), "within1 single %.3f -> mean %.3f | pts single %.2f -> mean %.2f | wrong %d -> %d" % (
                  (a.single<=1).mean(), (a.err<=1).mean(), pts(a.single.values).mean(), pts(a.err.values).mean(), (a.single>8).sum(), (a.err>8).sum()))

## Kill rule

New set above 800 m: share within 1 sigma at least 0.40 and no new wrong answer; official and NUDT: no loss of points per photo. See the tables above: compare each variant's `within1`, `wrong`, `pts_per_photo` with `c`.